In [ ]:
import glob, os, csv, json, orjson
from tqdm import tqdm
import pandas as pd
from tableone import TableOne


In [10]:
snowmed_map = pd.read_csv("snowmed_map.csv")
# Convert the SNOWMED CT Code to string
snowmed_map["SNOMED CT Code"] = snowmed_map["SNOMED CT Code"].astype(str)
snowmed_map = snowmed_map.set_index("SNOMED CT Code").to_dict(orient="index")
snowmed_map

{'270492004': {'Dx': '1st degree av block', 'Abbreviation': 'IAVB'},
 '195042002': {'Dx': '2nd degree av block', 'Abbreviation': 'IIAVB'},
 '164951009': {'Dx': 'abnormal QRS', 'Abbreviation': 'abQRS'},
 '426664006': {'Dx': 'accelerated junctional rhythm', 'Abbreviation': 'AJR'},
 '57054005': {'Dx': 'acute myocardial infarction', 'Abbreviation': 'AMI'},
 '413444003': {'Dx': 'acute myocardial ischemia', 'Abbreviation': 'AMIs'},
 '426434006': {'Dx': 'anterior ischemia', 'Abbreviation': 'AnMIs'},
 '54329005': {'Dx': 'anterior myocardial infarction', 'Abbreviation': 'AnMI'},
 '251173003': {'Dx': 'atrial bigeminy', 'Abbreviation': 'AB'},
 '164889003': {'Dx': 'atrial fibrillation', 'Abbreviation': 'AF'},
 '195080001': {'Dx': 'atrial fibrillation and flutter',
  'Abbreviation': 'AFAFL'},
 '164890007': {'Dx': 'atrial flutter', 'Abbreviation': 'AFL'},
 '195126007': {'Dx': 'atrial hypertrophy', 'Abbreviation': 'AH'},
 '251268003': {'Dx': 'atrial pacing pattern', 'Abbreviation': 'AP'},
 '713422000

In [ ]:
with open("/sailhome/kelvinkn/scr2_juice/code15_final_results.json", "r") as f:
    data = orjson.loads(f.read())

# with open("code15_test.json", "w", encoding="utf-8") as f:
#     f.write(orjson.dumps(data, option=orjson.OPT_INDENT_2).decode("utf-8"))

In [3]:
classes_map = {
    "RBBB":   "59118001",
    "LBBB":   "164909002",
    "SB":     "426177001",
    "ST":     "427084000",
    "AF":     "164889003",
    "1dAVb":  "270492004"
}

# Computing Accuracy of Prna

In [4]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def extract_label_pairs(records, classes_map):
    y_true = []
    y_pred = []

    for rec in records:
        snowmed = rec.get("snowmed_vals", {})
        for label, code in classes_map.items():
            if code in snowmed:
                entry = snowmed[code]
                if "present" in entry and "actual_present" in entry:
                    y_true.append(entry["actual_present"])
                    y_pred.append(entry["present"])
    return y_true, y_pred

# Assume records is already loaded (e.g. from previous step)
# And classes_map is defined
y_true, y_pred = extract_label_pairs(data, classes_map)

# Overall metrics
acc = accuracy_score(y_true, y_pred)
prec, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")

print("=== Classification Metrics ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")


=== Classification Metrics ===
Accuracy:  0.9449
Precision: 0.2272
Recall:    0.9111
F1 Score:  0.3637


In [16]:
report = classification_report(y_true, y_pred, target_names=["Negative", "Positive"])
print(report)


              precision    recall  f1-score   support

    Negative       1.00      0.95      0.97   2019643
    Positive       0.23      0.91      0.36     35537

    accuracy                           0.94   2055180
   macro avg       0.61      0.93      0.67   2055180
weighted avg       0.99      0.94      0.96   2055180



In [17]:
with open("table1/classification_metrics.txt", "w") as f:
    f.write("=== Classification Metrics (Binary) ===\n")
    f.write(f"Accuracy:  {acc:.4f}\n")
    f.write(f"Precision: {prec:.4f}\n")
    f.write(f"Recall:    {recall:.4f}\n")
    f.write(f"F1 Score:  {f1:.4f}\n\n")
    f.write("=== Classification Report ===\n")
    f.write(report)

print(f"✅ Saved classification metrics to table1/classification_metrics.txt")

✅ Saved classification metrics to table1/classification_metrics.txt


In [6]:
import pandas as pd

def compute_per_label_metrics(records, classes_map):
    results = []

    for label, code in classes_map.items():
        y_true = []
        y_pred = []

        for rec in records:
            snowmed = rec.get("snowmed_vals", {})
            if code in snowmed:
                entry = snowmed[code]
                if "present" in entry and "actual_present" in entry:
                    y_true.append(entry["actual_present"])
                    y_pred.append(entry["present"])

        if y_true:  # Only compute if there are valid entries
            acc = accuracy_score(y_true, y_pred)
            prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
            results.append({
                "label": label,
                "snomed_code": code,
                "n_samples": len(y_true),
                "accuracy": acc,
                "precision": prec,
                "recall": rec,
                "f1": f1
            })

    return pd.DataFrame(results)
label_metrics_df = compute_per_label_metrics(data, classes_map)
label_metrics_df 

,label,snomed_code,n_samples,accuracy,precision,recall,f1
0,RBBB,59118001,342530,0.927703,0.265491,0.909972,0.411054
1,LBBB,164909002,342530,0.986804,0.581973,0.869362,0.697213
2,SB,426177001,342530,0.853587,0.096874,0.964016,0.176056
3,ST,427084000,342530,0.979564,0.521207,0.917779,0.664847
4,AF,164889003,342530,0.981356,0.522866,0.899004,0.661184
5,1dAVb,270492004,342530,0.940201,0.000000,0.000000,0.000000


In [14]:
label_metrics_df.to_csv("table1/code15_label_metrics.csv", index=False)

# Chagas Analysis

## Prna's predicted snowmed codes

In [8]:
from tableone import TableOne

def records_to_tableone_df(records, classes_map):
    rows = []

    for rec in records:
        row = {"chagas": rec.get("chagas", False)}
        for label, code in classes_map.items():
            val = rec.get("snowmed_vals", {}).get(code)
            if val:
                row[label] = val.get("actual_present", False)
            else:
                row[label] = None  # missing condition
        rows.append(row)

    return pd.DataFrame(rows)

table1_full_snowmed_df = records_to_tableone_df(data, classes_map)

# Define categorical variables
categorical = list(classes_map.keys())

# Create the table
table1 = TableOne(data=table1_full_snowmed_df, columns=categorical, categorical=categorical, groupby="chagas", pval=True)

# Display or export
print(table1)

                   Grouped by chagas                                                      
                             Missing         Overall           False          True P-Value
n                                             342530          335978          6552        
RBBB, n (%)  False                     333033 (97.2)   327791 (97.6)   5242 (80.0)  <0.001
             True                         9497 (2.8)      8187 (2.4)   1310 (20.0)        
LBBB, n (%)  False                     336544 (98.3)   330215 (98.3)   6329 (96.6)  <0.001
             True                         5986 (1.7)      5763 (1.7)     223 (3.4)        
SB, n (%)    False                     336972 (98.4)   330628 (98.4)   6344 (96.8)  <0.001
             True                         5558 (1.6)      5350 (1.6)     208 (3.2)        
ST, n (%)    False                     334965 (97.8)   328516 (97.8)   6449 (98.4)  <0.001
             True                         7565 (2.2)      7462 (2.2)     103 (1.6)        

In [13]:
table1.to_csv("table1/table1_prna_snowmed.csv", index=False)

## All Snowmed's codes

In [18]:
def flatten_records_for_tableone(records):
    data = []

    for rec in records:
        row = {"chagas": rec.get("chagas", False)}
        for code, val in rec.get("snowmed_vals", {}).items():
            abbrev = snowmed_map[code]['Abbreviation']
            if "actual_present" in val:
                present = val["actual_present"]
                row[f"{abbrev}_ACTUAL"] = present
            present = val.get("present")
            row[f"{abbrev}_PRED"] = present
        data.append(row)

    return pd.DataFrame(data)

table1_flattened_df = flatten_records_for_tableone(data)

# All SNOMED columns (those starting with 'SNOMED_') are categorical
snomed_cols = [col for col in table1_flattened_df.columns if col.endswith("ACTUAL") or col.endswith("PRED")]

# Generate the table
table1_all_snowmed = TableOne(data=table1_flattened_df, columns=snomed_cols, categorical=snomed_cols, groupby="chagas", pval=True)


# table1_all_snowmed.to_csv("table1_chagas_vs_all_snomed.csv")



In [19]:
print(table1_all_snowmed)

                          Grouped by chagas                                                      
                                    Missing         Overall           False          True P-Value
n                                                    342530          335978          6552        
PR_PRED, n (%)      False                     342055 (99.9)   335593 (99.9)   6462 (98.6)  <0.001
                    True                          475 (0.1)       385 (0.1)      90 (1.4)        
LQT_PRED, n (%)     False                     328728 (96.0)   322463 (96.0)   6265 (95.6)   0.154
                    True                        13802 (4.0)     13515 (4.0)     287 (4.4)        
AF_ACTUAL, n (%)    False                     335599 (98.0)   329450 (98.1)   6149 (93.8)  <0.001
                    True                         6931 (2.0)      6528 (1.9)     403 (6.2)        
AF_PRED, n (%)      False                     330613 (96.5)   324822 (96.7)   5791 (88.4)  <0.001
                    

In [20]:
table1_all_snowmed.to_csv("table1_chagas_vs_all_snomed.csv")

In [10]:
print(table1_all_snowmed)

                              Grouped by chagas                                                      
                                        Missing         Overall           False          True P-Value
n                                                        342530          335978          6552        
SNOMED_10370003, n (%)  False                     342055 (99.9)   335593 (99.9)   6462 (98.6)  <0.001
                        True                          475 (0.1)       385 (0.1)      90 (1.4)        
SNOMED_111975006, n (%) False                     328728 (96.0)   322463 (96.0)   6265 (95.6)   0.154
                        True                        13802 (4.0)     13515 (4.0)     287 (4.4)        
SNOMED_164889003, n (%) False                     335599 (98.0)   329450 (98.1)   6149 (93.8)  <0.001
                        True                         6931 (2.0)      6528 (1.9)     403 (6.2)        
SNOMED_164890007, n (%) False                     340688 (99.5)   334242 (99.5)   

,Unnamed: 0,Unnamed: 1,Grouped by chagas,Grouped by chagas.1,Grouped by chagas.2,Grouped by chagas.3,Grouped by chagas.4
0,NaN,NaN,Missing,Overall,False,True,P-Value
1,n,NaN,NaN,342530,335978,6552,NaN
2,"SNOMED_10370003, n (%)",False,NaN,342055 (99.9),335593 (99.9),6462 (98.6),<0.001
3,"SNOMED_10370003, n (%)",True,NaN,475 (0.1),385 (0.1),90 (1.4),NaN
4,"SNOMED_111975006, n (%)",False,NaN,328728 (96.0),322463 (96.0),6265 (95.6),0.154
5,"SNOMED_111975006, n (%)",True,NaN,13802 (4.0),13515 (4.0),287 (4.4),NaN
6,"SNOMED_164889003, n (%)",False,NaN,330613 (96.5),324822 (96.7),5791 (88.4),<0.001
7,"SNOMED_164889003, n (%)",True,NaN,11917 (3.5),11156 (3.3),761 (11.6),NaN
8,"SNOMED_164890007, n (%)",False,NaN,340688 (99.5),334242 (99.5),6446 (98.4),<0.001
9,"SNOMED_164890007, n (%)",True,NaN,1842 (0.5),1736 (0.5),106 (1.6),NaN


In [5]:
table1_all_snowmed.to_csv("/sailhome/kelvinkn/scr2_juice/other_work/edwards/table1/table1_chagas_all_snowmed_no_ground_truth.csv")

In [12]:
import os
os.getcwd()

'/juice2/scr2/kelvinkn'